# 01 - Initial Data Exploration
## Credit Risk Intelligence — Canadian Banking Project

## 1. Libraries & Setup

In [2]:
# Import core data manipulation library 
import pandas as pd

# Load trainging dataset and drop duplicate index column
df = pd.read_csv('../data/cs-training.csv', index_col=0)

# Previews first rows
df.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


## 2. Dataset Overview

In [3]:
# Check dataset dimensions
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 150000
Columns: 11


In [4]:
# check column data types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 1 to 150000
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   SeriousDlqin2yrs                      150000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 2   age                                   150000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 4   DebtRatio                             150000 non-null  float64
 5   MonthlyIncome                         120269 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 7   NumberOfTimes90DaysLate               150000 non-null  int64  
 8   NumberRealEstateLoansOrLines          150000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 10  NumberOfDependents                    146076 non-null  float64
dtypes: float64(

## 3. Missing Values Analysis

In [5]:
# Check for missing values count and percentage per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
               
missing_df = pd.DataFrame({
    'Missing Values': missing,
    'Percentage (%)': missing_pct
})

#show only columns with missing values
missing_df[missing_df['Missing Values'] > 0] 

,Missing Values,Percentage (%)
MonthlyIncome,29731,19.82
NumberOfDependents,3924,2.62


In [6]:
# Deleting rows with missing values in 'NumberOfDependents' column
df = df.dropna(subset=['NumberOfDependents'])

# Fill missing values in 'MonthlyIncome' column with median value
median_income = df['MonthlyIncome'].median()
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(median_income)

# Check for missing values count per column
df.isnull().sum()

SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64

In [7]:
# check basic statistics for all columns
df.describe()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,146076.000000,146076.000000,146076.000000,146076.000000,146076.000000,1.460760e+05,146076.000000,146076.000000,146076.000000,146076.000000,146076.000000
mean,0.067410,5.922272,52.099277,0.407945,333.373603,6.445813e+03,8.529279,0.250698,1.029717,0.225027,0.757222
std,0.250732,250.070774,14.604005,4.002747,1943.906679,1.306129e+04,5.149533,3.977197,1.132774,3.962048,1.115086
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.031018,41.000000,0.000000,0.171764,3.820000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.158818,52.000000,0.000000,0.357751,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,0.000000,0.563684,62.000000,0.000000,0.766117,7.500000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,1.000000,50708.000000,107.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


In [8]:
#check number of rows with age equal to 0
age_zero = (df['age'] == 0).sum()
print(f'Numbers of rows with age equal to 0: {age_zero}')

Numbers of rows with age equal to 0: 1


In [9]:
# Exclude rows with age equal to 0
df = df[df['age'] > 0]
print(f'Numbers of rows with age equal to 0: {(df["age"] == 0).sum()}')

Numbers of rows with age equal to 0: 0


In [10]:
# Check number of rows with suspicious value 98 in late payment columns
late_payment_cols = ['NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfTimes90DaysLate']
for col in late_payment_cols:
    count_98 = (df[col] == 98).sum()
    print(f'Number of rows with value 98 in {col}: {count_98}')

Number of rows with value 98 in NumberOfTime30-59DaysPastDueNotWorse: 233
Number of rows with value 98 in NumberOfTime60-89DaysPastDueNotWorse: 233
Number of rows with value 98 in NumberOfTimes90DaysLate: 233


In [11]:
# Remove rows with invalid value 98 in late payment columns (likely a data encoding error)
df = df[(df[late_payment_cols] < 98).all(axis=1)]

print(df.shape)

(145842, 11)


In [12]:
df.describe()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,145842.000000,145842.000000,145842.000000,145842.000000,145842.000000,1.458420e+05,145842.000000,145842.000000,145842.000000,145842.000000,145842.000000
mean,0.066620,5.930170,52.126431,0.252026,333.900008,6.450327e+03,8.542909,0.094534,1.031356,0.068821,0.757841
std,0.249364,250.271233,14.590080,0.898449,1945.420869,1.307087e+04,5.142373,0.744209,1.132931,0.652193,1.115373
min,0.000000,0.000000,21.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.030925,41.000000,0.000000,0.172675,3.830000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.157999,52.000000,0.000000,0.358209,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,0.000000,0.560268,62.000000,0.000000,0.767213,7.500000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,1.000000,50708.000000,107.000000,96.000000,329664.000000,3.008750e+06,58.000000,96.000000,54.000000,96.000000,20.000000


In [13]:
# Check number of rows with suspicious value 96 in late payment columns
for col in late_payment_cols:
    count_96 = (df[col] == 96).sum()
    print(f'Number of rows with value 96 in {col}: {count_96}')

Number of rows with value 96 in NumberOfTime30-59DaysPastDueNotWorse: 5
Number of rows with value 96 in NumberOfTime60-89DaysPastDueNotWorse: 5
Number of rows with value 96 in NumberOfTimes90DaysLate: 5


In [14]:
# Remove rows with invalid value 96 in late payment columns (likely a data encoding error)
df = df[(df[late_payment_cols] < 96).all(axis=1)]

print(df.shape)
df.describe()

(145837, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,145837.000000,145837.000000,145837.000000,145837.000000,145837.000000,1.458370e+05,145837.000000,145837.000000,145837.000000,145837.000000,145837.000000
mean,0.066595,5.930339,52.127067,0.248743,333.911456,6.450408e+03,8.543202,0.091246,1.031391,0.065532,0.757846
std,0.249320,250.275522,14.589872,0.702078,1945.453236,1.307108e+04,5.142218,0.488371,1.132934,0.331422,1.115382
min,0.000000,0.000000,21.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.030924,41.000000,0.000000,0.172690,3.830000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.157984,52.000000,0.000000,0.358214,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,0.000000,0.560195,62.000000,0.000000,0.767233,7.500000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,1.000000,50708.000000,107.000000,13.000000,329664.000000,3.008750e+06,58.000000,17.000000,54.000000,11.000000,20.000000


In [15]:
# Convert 'age' column to integer type
df[['age', 'NumberOfDependents']] = df[['age', 'NumberOfDependents']].astype(int)

df[['age', 'NumberOfDependents']].dtypes

age                   int64
NumberOfDependents    int64
dtype: object

In [16]:
# Check 99th percentile for outlier columns
print(df['RevolvingUtilizationOfUnsecuredLines'].quantile(0.99))
print(df['DebtRatio'].quantile(0.99))

1.0932238110799988
4932.27999999997


In [18]:
# Remove extreme outliers above 99th percentile for utilization and debt ratio columns
p99_revolving = df['RevolvingUtilizationOfUnsecuredLines'].quantile(0.99)
p99_debt = df['DebtRatio'].quantile(0.99)

df = df[(df['RevolvingUtilizationOfUnsecuredLines'] <= p99_revolving) & (df['DebtRatio'] <= p99_debt)]

print(df.shape)
df.describe()

(142929, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,142929.000000,142929.000000,142929.000000,142929.000000,142929.000000,1.429290e+05,142929.000000,142929.000000,142929.000000,142929.000000,142929.000000
mean,0.063171,0.313359,52.184686,0.240581,253.144600,6.474797e+03,8.529144,0.083839,1.017225,0.061100,0.758453
std,0.243272,0.344067,14.619853,0.689012,760.960483,1.319323e+04,5.123206,0.459610,1.108404,0.316809,1.115355
min,0.000000,0.000000,21.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.030141,41.000000,0.000000,0.170915,3.812000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.152694,52.000000,0.000000,0.354470,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,0.000000,0.541131,62.000000,0.000000,0.734151,7.500000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,1.000000,1.093178,107.000000,13.000000,4931.000000,3.008750e+06,58.000000,17.000000,54.000000,11.000000,20.000000


In [27]:
# Save cleaned dataset to data folder
df.to_csv('../data/cs-training-cleaned.csv', index=False)